In [ ]:
!pip install pygam -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet, ElasticNetCV
from sklearn.pipeline import Pipeline

from pygam import LinearGAM, s

import itertools, warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from google.colab import files
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_excel(fname) if fname.lower().endswith(('.xlsx', '.xls')) else pd.read_csv(fname)

df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()

In [ ]:
df.info()
# Ukuran pusat & sebaran lengkap
desc = df.describe().T
desc['median']   = df.median()
desc['mode']     = df.mode().iloc[0]
desc['IQR']      = df.quantile(0.75) - df.quantile(0.25)
desc['skewness'] = df.skew()
desc['kurtosis'] = df.kurt()
desc

In [ ]:
print("Missing values per kolom:")
print(df.isnull().sum())
print("\nJumlah baris duplikat:", df.duplicated().sum())

In [ ]:
# Distribusi tiap variabel
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, df.columns):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribusi {col}')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot per variabel (deteksi outlier visual)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, df.columns):
    sns.boxplot(y=df[col], ax=ax, color='lightcoral')
    ax.set_title(col)
plt.tight_layout()
plt.show()

def iqr_outlier_count(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lb) | (s > ub)).sum()

print("Jumlah outlier (aturan IQR 1.5x) per kolom:")
print(df.apply(iqr_outlier_count))

In [ ]:
# Korelasi antar variabel
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriks Korelasi')
plt.show()

In [ ]:
# Hubungan tiap fitur terhadap target (regresi linear sederhana per fitur)
target = 'PE'
features = [c for c in df.columns if c != target]

fig, axes = plt.subplots(1, len(features), figsize=(18, 4))
for ax, col in zip(axes, features):
    sns.regplot(x=df[col], y=df[target], ax=ax,
                scatter_kws={'alpha': 0.15, 's': 10}, line_kws={'color': 'red'})
    ax.set_title(f'{col} vs {target}')
plt.tight_layout()
plt.show()

In [ ]:
# Multikolinearitas (VIF)
X_check = sm.add_constant(df[features])
vif_df = pd.DataFrame({
    'variabel': X_check.columns,
    'VIF': [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])]
})
vif_df

In [ ]:
# Model linear awal (baseline) untuk cek residual
model0 = smf.ols(f'{target} ~ ' + ' + '.join(features), data=df).fit()
resid = model0.resid

# Normalitas residual
sample_resid = resid.sample(min(5000, len(resid)), random_state=RANDOM_STATE)
shapiro_stat, shapiro_p = stats.shapiro(sample_resid)
print(f"Shapiro-Wilk: statistic={shapiro_stat:.4f}, p-value={shapiro_p:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(resid, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Residual')
sm.qqplot(resid, line='45', ax=axes[1])
axes[1].set_title('QQ-Plot Residual')
plt.tight_layout()
plt.show()

# Homoskedastisitas
bp_stat, bp_p, _, _ = het_breuschpagan(resid, model0.model.exog)
print(f"Breusch-Pagan: statistic={bp_stat:.4f}, p-value={bp_p:.4g}")
print("(p-value kecil -> indikasi heteroskedastisitas, perlu dicatat sebagai catatan interpretasi, bukan alasan transformasi otomatis)")

In [ ]:
X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

# Skema CV yang SAMA dipakai di semua model agar perbandingan adil
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

def compute_aic(n, rss, k):
    return n * np.log(rss / n) + 2 * k

def compute_aicc(n, rss, k):
    aic = compute_aic(n, rss, k)
    if n - k - 1 > 0:
        aic += (2 * k * (k + 1)) / (n - k - 1)
    return aic

In [ ]:
poly_results = []
for degree in [1, 2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    Xp_train = poly.fit_transform(X_train)
    ols = sm.OLS(y_train.values, sm.add_constant(Xp_train)).fit()

    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr', LinearRegression())
    ])
    cv_mse = -cross_val_score(pipe, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    poly_results.append({
        'degree': degree,
        'n_params': Xp_train.shape[1] + 1,
        'AIC': ols.aic,
        'CV_RMSE': np.sqrt(cv_mse.mean())
    })

poly_df = pd.DataFrame(poly_results)
poly_df

In [ ]:
best_deg_aic = poly_df.loc[poly_df['AIC'].idxmin(), 'degree']
best_deg_cv  = poly_df.loc[poly_df['CV_RMSE'].idxmin(), 'degree']
print(f"Derajat terpilih via AIC : {best_deg_aic}")
print(f"Derajat terpilih via CV  : {best_deg_cv}")

rho_poly, p_poly = spearmanr(poly_df['AIC'], poly_df['CV_RMSE'])
print(f"Korelasi Spearman ranking AIC vs CV_RMSE: rho={rho_poly:.3f}, p={p_poly:.3f}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
n_train = X_train_scaled.shape[0]

alphas = np.logspace(-4, 1, 25)
l1_ratios = [0.1, 0.5, 0.7, 0.9, 1.0]

en_results = []
for l1r in l1_ratios:
    for a in alphas:
        m = ElasticNet(alpha=a, l1_ratio=l1r, max_iter=10000, random_state=RANDOM_STATE)
        m.fit(X_train_scaled, y_train)
        rss = np.sum((y_train.values - m.predict(X_train_scaled)) ** 2)
        k = np.sum(m.coef_ != 0) + 1
        en_results.append({'l1_ratio': l1r, 'alpha': a, 'k': k,
                            'AICc': compute_aicc(n_train, rss, k)})

en_df = pd.DataFrame(en_results)
best_aicc_en = en_df.loc[en_df['AICc'].idxmin()]
print("Elastic Net terbaik via AICc:")
print(best_aicc_en)

In [ ]:
en_cv = ElasticNetCV(l1_ratio=l1_ratios, alphas=alphas, cv=kf,
                     max_iter=10000, random_state=RANDOM_STATE)
en_cv.fit(X_train_scaled, y_train)
print(f"Elastic Net terbaik via CV: alpha={en_cv.alpha_:.5f}, l1_ratio={en_cv.l1_ratio_}")

In [ ]:
# CV_MSE untuk SETIAP kombinasi (alpha, l1_ratio) agar bisa dikorelasikan dengan AICc
# (catatan: loop ini menjalankan ~125 x 10-fold CV, mungkin perlu beberapa menit)
cv_mse_en = []
for l1r in l1_ratios:
    for a in alphas:
        m = ElasticNet(alpha=a, l1_ratio=l1r, max_iter=10000, random_state=RANDOM_STATE)
        scores = cross_val_score(m, X_train_scaled, y_train, cv=kf, scoring='neg_mean_squared_error')
        cv_mse_en.append(-scores.mean())

en_df['CV_MSE'] = cv_mse_en
rho_en, p_en = spearmanr(en_df['AICc'], en_df['CV_MSE'])
print(f"Korelasi Spearman AICc vs CV_MSE (Elastic Net, semua kombinasi): rho={rho_en:.3f}, p={p_en:.3f}")

In [ ]:
X_train_arr = X_train.values
y_train_arr = y_train.values
lams = np.logspace(-3, 3, 15)

gam_gcv = LinearGAM(s(0) + s(1) + s(2) + s(3)).gridsearch(X_train_arr, y_train_arr, lam=lams, progress=False)

edof = gam_gcv.statistics_['edof']
rss_gam = np.sum((y_train_arr - gam_gcv.predict(X_train_arr)) ** 2)
aic_gam_gcv = compute_aic(len(y_train_arr), rss_gam, edof)

print("Lambda terpilih via GCV :", gam_gcv.lam)
print(f"Effective DoF: {edof:.2f} | AIC (df=edof): {aic_gam_gcv:.2f}")

In [ ]:
# CV manual per nilai lambda (skema kf yang sama, agar sebanding dgn model lain)
gam_cv_results = []
for lam in lams:
    fold_mse = []
    for tr_idx, val_idx in kf.split(X_train_arr):
        g = LinearGAM(s(0) + s(1) + s(2) + s(3), lam=lam).fit(
            X_train_arr[tr_idx], y_train_arr[tr_idx])
        pred = g.predict(X_train_arr[val_idx])
        fold_mse.append(np.mean((y_train_arr[val_idx] - pred) ** 2))
    gam_cv_results.append({'lam': lam, 'CV_MSE': np.mean(fold_mse)})

gam_cv_df = pd.DataFrame(gam_cv_results)
best_lam_cv = gam_cv_df.loc[gam_cv_df['CV_MSE'].idxmin(), 'lam']
print("Lambda terpilih via CV manual:", best_lam_cv)
print("Lambda terpilih via GCV      :", gam_gcv.lam)

In [ ]:
X_eng = X_train.copy()
for c in features:
    X_eng[f'{c}_sq'] = X_train[c] ** 2
for c1, c2 in itertools.combinations(features, 2):
    X_eng[f'{c1}_x_{c2}'] = X_train[c1] * X_train[c2]

candidate_terms = list(X_eng.columns)
print("Jumlah kandidat term:", len(candidate_terms))
print(candidate_terms)

In [ ]:
def forward_stepwise_aic(X_full, y, candidates):
    selected, remaining = [], list(candidates)
    current_aic = np.inf
    history = []
    while remaining:
        best_term, best_aic = None, current_aic
        for term in remaining:
            trial = selected + [term]
            model = sm.OLS(y.values, sm.add_constant(X_full[trial])).fit()
            if model.aic < best_aic:
                best_aic, best_term = model.aic, term
        if best_term is None:
            break
        selected.append(best_term)
        remaining.remove(best_term)
        current_aic = best_aic
        history.append({'step': len(selected), 'added': best_term, 'AIC': current_aic})
    return selected, pd.DataFrame(history)

selected_terms_aic, hist_aic = forward_stepwise_aic(X_eng, y_train, candidate_terms)
print("Term terpilih (forward stepwise, kriteria AIC):")
print(selected_terms_aic)
hist_aic

In [ ]:
def forward_stepwise_cv(X_full, y, candidates, kf):
    selected, remaining = [], list(candidates)
    current_score = np.inf
    history = []
    while remaining:
        best_term, best_score = None, current_score
        for term in remaining:
            trial = selected + [term]
            Xc = X_full[trial].values
            fold_mse = []
            for tr_idx, val_idx in kf.split(Xc):
                lr = LinearRegression().fit(Xc[tr_idx], y.values[tr_idx])
                pred = lr.predict(Xc[val_idx])
                fold_mse.append(np.mean((y.values[val_idx] - pred) ** 2))
            score = np.mean(fold_mse)
            if score < best_score:
                best_score, best_term = score, term
        if best_term is None:
            break
        selected.append(best_term)
        remaining.remove(best_term)
        current_score = best_score
        history.append({'step': len(selected), 'added': best_term, 'CV_MSE': current_score})
    return selected, pd.DataFrame(history)

selected_terms_cv, hist_cv = forward_stepwise_cv(X_eng, y_train, candidate_terms, kf)
print("Term terpilih (forward stepwise, kriteria CV):")
print(selected_terms_cv)
hist_cv

In [ ]:
summary = pd.DataFrame([
    {
        'Model': 'Polynomial (nested)',
        'Pilihan_AIC': f"degree={best_deg_aic}",
        'Pilihan_CV': f"degree={best_deg_cv}",
        'Sama?': best_deg_aic == best_deg_cv
    },
    {
        'Model': 'Elastic Net',
        'Pilihan_AIC': f"alpha={best_aicc_en['alpha']:.4f}, l1_ratio={best_aicc_en['l1_ratio']}",
        'Pilihan_CV': f"alpha={en_cv.alpha_:.4f}, l1_ratio={en_cv.l1_ratio_}",
        'Sama?': np.isclose(best_aicc_en['alpha'], en_cv.alpha_, rtol=0.2) and best_aicc_en['l1_ratio'] == en_cv.l1_ratio_
    },
    {
        'Model': 'GAM',
        'Pilihan_AIC': f"lambda≈{gam_gcv.lam}",
        'Pilihan_CV': f"lambda≈{best_lam_cv}",
        'Sama?': np.isclose(np.ravel(gam_gcv.lam), best_lam_cv, rtol=0.2).all()
    },
    {
        'Model': 'Best Subset (forward stepwise)',
        'Pilihan_AIC': f"{len(selected_terms_aic)} term: {selected_terms_aic}",
        'Pilihan_CV': f"{len(selected_terms_cv)} term: {selected_terms_cv}",
        'Sama?': set(selected_terms_aic) == set(selected_terms_cv)
    },
])
pd.set_option('display.max_colwidth', 100)
summary

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# Contoh: evaluasi model polynomial pilihan AIC vs pilihan CV pada X_test
for label, deg in [('AIC', best_deg_aic), ('CV', best_deg_cv)]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr', LinearRegression())
    ]).fit(X_train, y_train)
    pred_test = pipe.predict(X_test)
    print(f"Polynomial (pilihan {label}, degree={deg}) -> Test RMSE = {rmse(y_test.values, pred_test):.4f}")

# Pola yang sama bisa diulang untuk Elastic Net, GAM, dan Best Subset
# dengan konfigurasi masing-masing yang sudah tersimpan di variabel di atas.